In [ ]:
%env JAX_PLATFORMS=cpu
%env JAX_ENABLE_X64=True
%env JAX_THREEFRY_PARTITIONABLE=False

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
root = next(
    (path for path in (cwd, *cwd.parents) if (path / "experiments/local_rules/local_rules.py").is_file()),
    None,
)
if root is None:
    raise RuntimeError("Open this notebook from the repository or its experiment folder.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from experiments.local_rules import local_rules as local
from spikegd.local_rules import paper, rflo as qb

import jax
import matplotlib.pyplot as plt

In [ ]:
plt.style.use("spikegd.utils.plotstyle")

# Run experiments

Run this notebook from a fresh kernel with the repository installed. The RFLO comparison uses one configuration and one seed. The notebook also lists the full sweep and includes an optional preview of the published spike-time ablations. See the [README](README.md) for the full runs.

Defaults: task `cycle2` (the two read-out neurons fire B C B C), `NET` with 3 neurons, the last 2 read out.

Per config there are three objects:

- `S = build(net, task)`: the system. Simulation, readout, loss, the exact gradient. Nothing in it is drawn per seed.
- `ini = make_init(S, seed)`: what is drawn per seed. $W^*$ (and $y^*$), $W_{\rm out}(0)$, $B(0)$, $\alpha(0)$ for every
  Jacobian form.
- `LocalRule(...)`: one rule. Its name is `RFLO_<what B, alpha come from>_<teacher corrections>@<Jacobian form>`, e.g.
  `RFLO_randB_0@murray` (random fixed B) or `RFLO_teachB_5@e` (B drawn by the teacher, corrected 5 times).

## Configuration

In [ ]:
DEMO_TRIALS = 10
INCLUDE_TEACHER = True
FULL_TEACHER_SWEEP = False
RUN_PAPER_PREVIEW = False

print("Optimizer", qb.BENCH["optimizer"])
print("Network", qb.NET)
print("Network ablations", qb.BENCH["net"])
print("Tasks", {name: qb.TASKS[name]["n_trials"] for name in qb.BENCH["tasks"]})
print("Teacher ablations", qb.BENCH["teacher"])
print("Rules", [rule.name for rule in qb.BENCH["rules"]])
print(f"{len(qb.configs())} configs, {len(qb.jobs())} jobs across {qb.BENCH['seeds']} seeds")

## Initialize the system

`describe(S)` prints the network, the input and the target sequence. `ini.describe(S)` prints what seed 0 draws.
The last line of the alignment is the starting point of feedback alignment: $\cos(B_{\rm targets}, W_{\rm out}(0))$.
If it is negative, a random-B rule starts by pushing the read-out neurons the wrong way.

In [ ]:
S = qb.build(qb.NET, qb.TASKS[qb.BASE_TASK])
qb.describe(S)
ini = qb.make_init(S, seed=0)
ini.describe(S)

out0 = S.simulate(S.w0)

## Compare gradients at initialization

(a) Credit check: $\sum_k W_{{\rm out},o}\,\ell(t_{ok})\,dt_{ok}/dw$ with $dt/dw$ from autodiff must equal the exact
gradient. If this fails, $\ell$ or the event bookkeeping is wrong and nothing below means anything.

(b) For every output of a read-out neuron $o$: the cosine between the student's time row $s_k$ and the exact
$dt_{ok}/dw_{o\cdot}$. This is the trace alone, with the correct credit, so it is the best a rule with this Jacobian
form can do on the read-out rows. Each form uses its $\alpha(0)$ from the init.

(c) The cosine of the full update with the exact gradient, per row, at init, for every rule with random B.

In [ ]:
local.inspect_gradients(S, ini, out0)

## Train the RFLO rules

A `LocalRule` has four parts:

| field | what it sets | values |
|---|---|---|
| `jacobian` | the own block $J^{ii}$: how neuron $i$'s trace evolves | `murray`, `e`, `exact_decay` (or one you add to `JACOBIANS`) |
| `cross` | the cross blocks $J^{iX}$ in the trace evolution | `none` ($J^{iX}=0$, as in RFLO) |
| `signal` | the factor $c_i$ in the learning signal $c_i\,\ell(t_k)$ | `B`, `W_out_targets`, `targets_only` |
| `B_from`, `alpha_from`, `P` | random and fixed, or corrected `P` times by the teacher | `random`, `teacher` |

A new Jacobian form is a function `f(S, ev, alpha) -> (s, traces)` added as
`qb.JACOBIANS["name"] = dict(fn=f, a_range=((lo, hi), (lo, hi)), fit=None, alpha_doc="...")`; after that
`qb.rflo("name", "teacher", 5)` is a rule. To bench it, add it to `qb.BENCH["rules"]` in `spikegd/local_rules/rflo.py`.

The runs below are short (`DEMO_TRIALS`) and one seed. Each printout says, in order: the final loss against the exact
gradient and against training only $W_{\rm out}$; the median cosine of the update with the exact gradient on the
read-out rows and on the hidden rows; how $\cos(B_{\rm targets},W_{\rm out})$ moved; and for the teacher, how far each
correction moved B and how many perturbed simulations it cost.

The default demonstration uses ten training trials and one teacher correction. Set
`DEMO_TRIALS = 2000` and `FULL_TEACHER_SWEEP = True` for the longer comparison with all three teacher rules.

In [ ]:
DEMO_RULES = [qb.EXACT, qb.READOUT_ONLY] + [qb.rflo(jacobian) for jacobian in qb.JACOBIANS]
if INCLUDE_TEACHER:
    if FULL_TEACHER_SWEEP:
        DEMO_RULES.extend(qb.rflo(jacobian, "teacher", 5) for jacobian in qb.JACOBIANS)
    else:
        DEMO_RULES.append(qb.rflo("murray", "teacher", 1, M=1, R=1))

DEMO = local.run_demo(S, ini, DEMO_RULES, n_trials=DEMO_TRIALS)

## RFLO ablation grid

`BENCH["grid"] = "minmidmax"` (the default). For each task, cycles / k = 0.25, 1 and 8 (`BENCH["mmm_tasks"]`), every
ablation axis is set to its smallest, middle and largest value, one axis at a time, with everything else at base:

- network: N, k/N (applied at the base N, so k = 1, 2, 3 for N = 3), w_scale, w_in, r_in;
- teacher (teacher-B rules only): P / trials, R / phase, M.

A value that equals base is not run again: it is the task's `base` config. Every config runs all the rules in
`BENCH["rules"]` (the teacher axes only the teacher-B rules), with 5 seeds. `grid = "full"` goes back to every value
of every axis. The grid settings are in `spikegd/local_rules/rflo.py`. Run from the repository root.

```sh
python -m experiments.local_rules.local_rules rflo-batch -j 8
python -m experiments.local_rules.local_rules rflo-batch -j 8 -f mmm.cyc_per_k=1.
python -m experiments.local_rules.local_rules rflo-batch -j 8 -f k_over_N
```

Finished jobs are skipped, so it can be stopped and started again. The cell below lists the configs.

In [ ]:
C = qb.configs()
J = qb.jobs()
print(f"grid '{qb.BENCH['grid']}': {len(C)} configs, {len(J)} jobs "
      f"({sum(r.teacher for *_, r, _ in J)} of them with a teacher)")
for cid, net, t, ov in C:
    n_rules = len(qb.BENCH["rules"]) if ov is None else sum(r.B_from == "teacher" for r in qb.BENCH["rules"])
    print(f"  {cid:38s} N={net['N']:2d} k={net['k']}  n_cycles={qb.TASKS[t]['n_cycles']:<5g} "
          f"trials={qb.TASKS[t]['n_trials']:6d}  rules={n_rules:2d}  " + (f"teacher {ov}" if ov else ""))

## Preview the published spike-time ablations

Enable this cell to compare exact and local updates on the published RNN spike-time task for four trials. The full workflow records exact runs, fits corrections, trains on jittered inputs and perturbed weights, and produces paper figures and tables. Its commands are in the [README](README.md#published-spike-time-ablations).

In [ ]:
if RUN_PAPER_PREVIEW:
    task, neuron = paper.make_task(), paper.make_neuron()
    for method in ("exact", "local"):
        step = paper.make_trainer(
            task, neuron, paper.make_rule(method, task, neuron),
            with_lyapunov=False, n_trials=4, probe_every=2,
        )
        record = jax.jit(step)(task["spikes_in"], task["w0"], task["x0"])
        print(method, record["loss"])

# Plot results

## Initial readout

In [ ]:
t_spk0, _ = S.target_times(out0)
fig, ax = plt.subplots(figsize=(9, 2.4))
ax.plot(S.GRID, ini.Y_star, "k", label="y*")
ax.plot(S.GRID, S.readout(ini.W_out0, t_spk0), label="y at init")
ax.set_xlabel("Time")
ax.legend(frameon=False)
plt.show()

## Training loss

In [ ]:
local.plot_demo_loss(DEMO)
plt.show()

## Teacher feedback

In [ ]:
if INCLUDE_TEACHER:
    local.plot_feedback(S, DEMO, DEMO_RULES)
    plt.show()